In [1]:
from dotenv import load_dotenv
import os
from typing import List, Dict, Any, Optional, Union
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "Notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


load_dotenv(ROOT / ".env")


# Import from our Classes module
from Classes.model_classes import SQLLineageExtractor, SQLLineageResult, create_sql_lineage_extractor
from Classes.validation_classes import SQLLineageValidator


MODEL = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
PROVIDER = "scaleway"
HF_TOKEN = os.environ.get("HF_TOKEN")

/Users/nikolajabramov/PycharmProjects/llm4lineage/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:27: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
# Pre-check: validate Hugging Face token before any model call
from huggingface_hub import HfApi

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add HF_TOKEN=... to .env (or export it), then restart the kernel."
    )

print(f"HF_TOKEN found: ...{HF_TOKEN[-6:]}")

try:
    whoami = HfApi().whoami(token=HF_TOKEN)
    print(f"Hugging Face auth OK for user: {whoami.get('name', '<unknown>')}")
except Exception as exc:
    raise RuntimeError(
        "HF_TOKEN is set but authentication failed. Verify token validity and access rights for the selected provider/model."
    ) from exc

HF_TOKEN found: ...OrZWkc
Hugging Face auth OK for user: Xpehutta


In [3]:
file_path = ROOT / "data" / "SQL.txt"



# Read file with example

with open(file_path, "r", encoding="utf-8") as f:

    SQL = f.read()

In [4]:

# Create extractor using factory function
extractor = create_sql_lineage_extractor(
    model=MODEL,
    provider=PROVIDER,
    hf_token=HF_TOKEN,
    max_new_tokens=2048,
    do_sample=False,
    max_retries=3,
    use_pydantic_parser=True
)


    
print("=" * 60)
print("SQL Lineage Extractor with langchain_huggingface")
print("=" * 60)

# Test connection
print(f"\nModel: {extractor.model}")
print(f"Provider: {extractor.provider}")

if extractor.test_connection():
    print("✓ Connection test successful")
else:
    print("✗ Connection test failed")

print(f"\nExtracting lineage from SQL ({len(SQL)} characters)...")

try:
    # Extract lineage
    result = extractor.extract(SQL)
    
    if "error" in result:
        print(f"✗ Error: {result['error']}")
    else:
        print("✓ Lineage extracted successfully!")
        print(f"\nTarget: {result.get('target', 'N/A')}")
        print(f"Sources ({result.get('source_count', len(result.get('sources', [])))}):")
        
        if result.get('sources'):
            for i, source in enumerate(result['sources'][:10], 1):  # Show first 10 sources
                print(f"  {i}. {source}")
            
            if len(result['sources']) > 10:
                print(f"  ... and {len(result['sources']) - 10} more")
        
        # Get as SQLLineageResult object
        lineage_result = extractor.extract_with_result(SQL)
        print(f"\nSQLLineageResult object:")
        #print(f"  String representation: {lineage_result}")
        print(f"  Source count: {lineage_result.source_count}")
        print(f"  As JSON: {lineage_result.to_json()}")

except Exception as e:
    print(f"\n✗ Unexpected error: {e}")
    import traceback
    traceback.print_exc()

SQL Lineage Extractor with langchain_huggingface

Model: Qwen/Qwen3-Coder-30B-A3B-Instruct
Provider: scaleway
✓ Connection test successful

Extracting lineage from SQL (17834 characters)...
✓ Lineage extracted successfully!

Target: s_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred
Sources (20):
  1. s_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred_tmp
  2. s_grnplm_as_t_didsd_010_vd_dwh.v_$eks_agrmnt_to_coa_3
  3. s_grnplm_as_t_didsd_010_vd_dwh.v_coa
  4. s_grnplm_as_t_didsd_010_vd_dwh.v_gl_main_acct
  5. s_grnplm_vd_t_bvd_db_dmslcl.a_agr_cred_coa_period
  6. s_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred_optn
  7. s_grnplm_as_t_didsd_010_vd_dwh.v_loan_agrmnt_rate
  8. s_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred_cust
  9. s_grnplm_as_t_didsd_029_vd_dwh.v_agr_cred
  10. s_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred_core_uvdo
  ... and 10 more

SQLLineageResult object:
  Source count: 21
  As JSON: {
  "target": "s_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred",
  "sources": [
    "s_grnplm_vd_t_bvd_db_dmslcl.d_agr_cred_tmp",
    "s_gr

In [5]:
# src/parsers/langchain_sql_parser.py
import os
import re
import asyncio
from typing import List, Dict, Any, Optional
from datetime import datetime
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_gigachat.chat_models import GigaChat


class LineageOutput(BaseModel):
    """Pydantic model for the expected lineage extraction output."""
    target_table: str = Field(description="Fully qualified target table/view name")
    source_tables: List[str] = Field(description="List of fully qualified source table names")
    join_conditions: List[str] = Field(description="Join conditions used")
    transformation_type: str = Field(description="Type of transformation (JOIN, UNION, etc.)")
    confidence: float = Field(description="Confidence score 0-1")


class LangChainSQLLineageExtractor:
    """
    SQL lineage extractor using GigaChat LLM and LangChain Expression Language (LCEL).
    No regex fallback – relies entirely on LLM.
    """

    def __init__(
        self,
        credentials: Optional[str] = None,
        model: str = "GigaChat",
        verify_ssl_certs: bool = False,
        scope: Optional[str] = None,
        base_url: Optional[str] = None,
        temperature: float = 0.1,
        max_tokens: int = 2048,
    ):
        """
        Initialize the extractor with GigaChat parameters.

        Args:
            credentials: GigaChat API key (mandatory; if None, reads from GIGACHAT_API_KEY env var).
            model: GigaChat model name (e.g., "GigaChat", "GigaChat-Pro").
            verify_ssl_certs: Whether to verify SSL certificates.
            scope: Optional scope (e.g., "GIGACHAT_API_PERS").
            base_url: Optional custom API endpoint.
            temperature: Generation temperature.
            max_tokens: Max tokens for response.
        """
        self.credentials = credentials or os.getenv("GIGACHAT_API_KEY")
        if not self.credentials:
            raise ValueError(
                "GigaChat API key must be provided via 'credentials' parameter "
                "or set in GIGACHAT_API_KEY environment variable."
            )

        self.model = model
        self.verify_ssl_certs = verify_ssl_certs
        self.scope = scope
        self.base_url = base_url
        self.temperature = temperature
        self.max_tokens = max_tokens

        # Set up output parser (Pydantic v2)
        self.output_parser = PydanticOutputParser(pydantic_object=LineageOutput)

        # Define prompt template
        self.prompt = PromptTemplate(
            template="""You are a SQL lineage extraction expert. Analyze the SQL statement and extract all source-to-target dependencies.

**SQL Statement:**
{sql_text}

**Extraction Rules:**
1. Identify the TARGET table/view (the main object being created/modified)
2. Extract ALL SOURCE tables/views referenced in:
   - FROM/JOIN clauses (including subqueries)
   - INSERT/UPDATE statements
   - CTEs that reference base tables
3. Remove quotes and aliases: "schema"."table" -> schema.table
4. Exclude: system tables, temp tables, CTE names, derived tables
5. Return fully qualified names (schema.table)
6. Extract join conditions where available

**Return Format:**
{format_instructions}

**Examples:**
SQL: CREATE VIEW sales.customer_summary AS SELECT c.* FROM sales.customers c
Output: {{
    "target_table": "sales.customer_summary",
    "source_tables": ["sales.customers"],
    "join_conditions": [],
    "transformation_type": "SELECT",
    "confidence": 0.5
}}

**Your Analysis:""",
            input_variables=["sql_text"],
            partial_variables={"format_instructions": self.output_parser.get_format_instructions()}
        )

        # Initialize GigaChat LLM
        self.llm = GigaChat(
            credentials=self.credentials,
            model=self.model,
            verify_ssl_certs=self.verify_ssl_certs,
            scope=self.scope,
            base_url=self.base_url,
            temperature=self.temperature,
            max_tokens=self.max_tokens,
            timeout=120
        )

        # Build the runnable chain using LCEL
        self.chain = self.prompt | self.llm | self.output_parser

    async def extract_lineage(self, sql_text: str) -> Dict[str, Any]:
        """
        Extract lineage from a single SQL statement using GigaChat.

        Args:
            sql_text: The SQL DDL or query string.

        Returns:
            Dictionary containing lineage information or an error structure.
        """
        try:
            cleaned_sql = self._clean_sql(sql_text)
            # Invoke the chain asynchronously
            lineage_obj: LineageOutput = await self.chain.ainvoke({"sql_text": cleaned_sql})
            # Convert Pydantic model to dict
            lineage_data = lineage_obj.model_dump()
            # Add metadata
            lineage_data["extraction_method"] = "gigachat_llm"
            lineage_data["extraction_timestamp"] = datetime.now().isoformat()
            lineage_data["sql_hash"] = hash(sql_text)
            return lineage_data
        except Exception as e:
            # Return an error structure if extraction fails
            return {
                "error": str(e),
                "extraction_method": "failed",
                "extraction_timestamp": datetime.now().isoformat(),
                "sql_hash": hash(sql_text)
            }

    def _clean_sql(self, sql_text: str) -> str:
        """Remove comments, normalize whitespace, and strip trailing semicolon."""
        # Remove single-line comments
        sql_text = re.sub(r'--.*$', '', sql_text, flags=re.MULTILINE)
        # Remove multi-line comments
        sql_text = re.sub(r'/\*.*?\*/', '', sql_text, flags=re.DOTALL)
        # Collapse multiple whitespace characters
        sql_text = re.sub(r'\s+', ' ', sql_text)
        # Remove trailing semicolon
        sql_text = sql_text.strip().rstrip(';')
        return sql_text.strip()

    def batch_extract(self, sql_texts: List[str]) -> List[Dict[str, Any]]:
        """
        Extract lineage from multiple SQL statements using asyncio.

        Args:
            sql_texts: List of SQL strings.

        Returns:
            List of lineage dictionaries (one per input SQL).
        """

        async def batch_process():
            tasks = [self.extract_lineage(sql) for sql in sql_texts]
            return await asyncio.gather(*tasks, return_exceptions=True)

        results = asyncio.run(batch_process())
        processed = []
        for res in results:
            if isinstance(res, Exception):
                # This case should not happen because extract_lineage catches exceptions,
                # but we keep it for safety.
                processed.append({"error": str(res)})
            else:
                processed.append(res)
        return processed